# Tutorial: Causal Discovery

This tutorial demonstrates **Causal Discovery** algorithms for learning causal structure from observational data:

1. **PC Algorithm** (constraint-based): Tests conditional independence, outputs CPDAG
2. **LiNGAM** (functional): Exploits non-Gaussianity via ICA, outputs unique DAG

**Key Concepts**:
- **Skeleton**: Undirected graph of adjacencies
- **V-structure**: Collider X → Z ← Y (orients edges)
- **CPDAG**: Markov equivalence class (indistinguishable from data)
- **Causal Order**: Topological ordering of variables

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../../src')

from causal_inference.discovery import (
    # Algorithms
    pc_algorithm, pc_skeleton, pc_orient,
    direct_lingam, ica_lingam, bootstrap_lingam,
    # Types
    Graph, DAG, CPDAG, PCResult, LiNGAMResult,
    # Utilities
    generate_random_dag, generate_dag_data,
    dag_to_cpdag, skeleton_f1, compute_shd,
    # Tests
    fisher_z_test, partial_correlation,
    check_non_gaussianity,
)

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

---

## Part 1: Why Causal Discovery?

### The Inverse Problem

Most causal inference assumes we **know** the causal graph:

$$X \rightarrow Y \leftarrow Z \quad \text{(given)}$$

**Causal discovery** solves the **inverse problem**: learn the graph from data.

### Two Paradigms

| Approach | Key Assumption | Output | Identifiability |
|----------|----------------|--------|------------------|
| **Constraint-based** (PC) | Faithfulness | CPDAG (equivalence class) | Up to Markov equivalence |
| **Functional** (LiNGAM) | Non-Gaussianity | Unique DAG | Fully identified |

### When Each is Appropriate

- **PC Algorithm**: Gaussian or unknown noise distribution; moderate sample size
- **LiNGAM**: Non-Gaussian data (confirmed); larger samples for ICA

### Generating Ground Truth

We use `generate_random_dag` to create known causal structures for validation.

In [ ]:
# Generate a 5-node DAG
true_dag = generate_random_dag(n_vars=5, edge_prob=0.4, seed=42)

print("=== Ground Truth DAG ===")
print(f"Nodes: {true_dag.n_nodes}")
print(f"Edges: {true_dag.n_edges}")
print(f"\nAdjacency Matrix:")
print(true_dag.adjacency_matrix)

# Show edges
print("\nEdges:")
for i, j in true_dag.edges():
    print(f"  X{i} → X{j}")

In [ ]:
def visualize_dag(dag, title="DAG"):
    """Simple DAG visualization using adjacency matrix."""
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    
    n = dag.n_nodes
    adj = dag.adjacency_matrix
    
    # Position nodes in a circle
    angles = np.linspace(0, 2*np.pi, n, endpoint=False) - np.pi/2
    x = np.cos(angles)
    y = np.sin(angles)
    
    # Draw edges
    for i in range(n):
        for j in range(n):
            if adj[i, j]:
                dx, dy = x[j] - x[i], y[j] - y[i]
                ax.arrow(x[i], y[i], dx*0.85, dy*0.85,
                        head_width=0.08, head_length=0.05, 
                        fc='gray', ec='gray', alpha=0.7)
    
    # Draw nodes
    ax.scatter(x, y, s=500, c='lightblue', edgecolors='navy', linewidth=2, zorder=5)
    for i in range(n):
        ax.annotate(f'X{i}', (x[i], y[i]), ha='center', va='center', fontsize=12, fontweight='bold')
    
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=14)
    plt.tight_layout()
    return fig

visualize_dag(true_dag, "Ground Truth DAG")
plt.show()

---

## Part 2: PC Algorithm Deep Dive

### Algorithm Overview

**Phase 1 (Skeleton)**:
1. Start with complete undirected graph
2. For each pair $(X, Y)$, test conditional independence: $X \perp\!\!\!\perp Y \mid S$
3. Remove edge if independent for any separator set $S$

**Phase 2 (Orientation)**:
1. Apply v-structure rule: if $X - Z - Y$ and $Z \notin \text{Sep}(X, Y)$, orient $X \to Z \leftarrow Y$
2. Apply Meek rules (R1-R4) to propagate orientations

### Key Assumption: Faithfulness

**Faithfulness**: All and only the independencies in the data reflect the DAG structure.

- Violations occur with probability zero (continuous distributions)
- Near-violations can cause errors with finite samples

### Example: Diamond Structure

Let's build a classic "diamond" DAG and trace through PC algorithm.

In [ ]:
# Create diamond: X0 → X1, X0 → X2, X1 → X3, X2 → X3
diamond_dag = DAG(4)
diamond_dag.add_edge(0, 1)  # X0 → X1
diamond_dag.add_edge(0, 2)  # X0 → X2
diamond_dag.add_edge(1, 3)  # X1 → X3
diamond_dag.add_edge(2, 3)  # X2 → X3 (collider at X3)

# Generate Gaussian data
data_diamond, B_diamond = generate_dag_data(
    diamond_dag, n_samples=1000, noise_type="gaussian", seed=42
)

print("=== Diamond Structure ===")
print("X0 → X1, X0 → X2, X1 → X3, X2 → X3")
print(f"\nData shape: {data_diamond.shape}")
print(f"True coefficient matrix B:\n{B_diamond}")

### Phase 1: Skeleton Learning

In [ ]:
# Run skeleton phase only
skeleton, sep_sets = pc_skeleton(data_diamond, alpha=0.01)

print("=== Learned Skeleton ===")
print(f"Edges found: {skeleton.n_edges}")
print("\nSkeleton adjacencies:")
for i, j in skeleton.edges():
    print(f"  X{i} — X{j}")

# Show separator sets (important for v-structure detection)
print("\nSeparator sets (used for orientation):")
for (i, j), sep in sep_sets.items():
    if i < j:  # Avoid duplicates
        print(f"  Sep(X{i}, X{j}) = {{{', '.join(f'X{k}' for k in sep)}}}")

In [ ]:
# Evaluate skeleton quality
precision, recall, f1 = skeleton_f1(skeleton, diamond_dag)

print("=== Skeleton Evaluation ===")
print(f"Precision: {precision:.3f} (edges found that are correct)")
print(f"Recall:    {recall:.3f} (true edges that were found)")
print(f"F1 Score:  {f1:.3f}")

### Phase 2: Orientation

The key insight: **V-structures** are identifiable from data!

In our diamond: $X_1 \to X_3 \leftarrow X_2$ is a v-structure because:
- $X_1 \not\perp\!\!\!\perp X_2$ marginally (both caused by $X_0$)
- But $X_3 \notin \text{Sep}(X_1, X_2)$

In [ ]:
# Run full PC algorithm
pc_result = pc_algorithm(data_diamond, alpha=0.01)

print("=== PC Algorithm Result ===")
print(f"Algorithm: {pc_result.algorithm}")
print(f"CI tests performed: {pc_result.n_ci_tests}")
print(f"Alpha: {pc_result.alpha}")

# Show CPDAG edges
cpdag = pc_result.cpdag
print("\nCPDAG edges:")
for i, j in cpdag.directed_edges():
    print(f"  X{i} → X{j} (directed)")
for i, j in cpdag.undirected_edges():
    print(f"  X{i} — X{j} (undirected)")

In [ ]:
# Compute Structural Hamming Distance
true_cpdag = dag_to_cpdag(diamond_dag)
shd = compute_shd(cpdag, true_cpdag)

print(f"=== Structural Hamming Distance ===")
print(f"SHD: {shd}")
print("\nInterpretation:")
print("  SHD = 0: Perfect recovery")
print("  SHD = 1: One edge difference")
print("  Higher: More structural errors")

---

## Part 3: PC Algorithm Diagnostics

### Sensitivity to Alpha (Significance Level)

The `alpha` parameter controls the trade-off:
- **Low alpha** (0.001): Conservative, may miss edges
- **High alpha** (0.10): Liberal, may add spurious edges

In [ ]:
# Test different alpha values
alphas = [0.001, 0.01, 0.05, 0.10]
results = []

print("=== Alpha Sensitivity ===")
print(f"{'Alpha':>8} {'Edges':>8} {'SHD':>8} {'F1':>8}")
print("-" * 35)

for alpha in alphas:
    result = pc_algorithm(data_diamond, alpha=alpha)
    shd = compute_shd(result.cpdag, true_cpdag)
    _, _, f1 = skeleton_f1(result.skeleton, diamond_dag)
    
    results.append({'alpha': alpha, 'shd': shd, 'f1': f1})
    n_edges = result.cpdag.n_directed_edges + result.cpdag.n_undirected_edges
    print(f"{alpha:>8.3f} {n_edges:>8} {shd:>8} {f1:>8.3f}")

print(f"\nTrue edges: {diamond_dag.n_edges}")

### Sample Size Effects

In [ ]:
# Test different sample sizes
sample_sizes = [100, 250, 500, 1000, 2000]
shd_by_n = []

print("=== Sample Size Effects ===")
print(f"{'N':>8} {'SHD':>8} {'Skeleton F1':>12}")
print("-" * 30)

for n in sample_sizes:
    data_n, _ = generate_dag_data(diamond_dag, n_samples=n, seed=42)
    result_n = pc_algorithm(data_n, alpha=0.01)
    shd = compute_shd(result_n.cpdag, true_cpdag)
    _, _, f1 = skeleton_f1(result_n.skeleton, diamond_dag)
    
    shd_by_n.append(shd)
    print(f"{n:>8} {shd:>8} {f1:>12.3f}")

# Plot
plt.figure(figsize=(8, 5))
plt.plot(sample_sizes, shd_by_n, 'o-', markersize=8)
plt.xlabel('Sample Size', fontsize=12)
plt.ylabel('Structural Hamming Distance', fontsize=12)
plt.title('PC Algorithm: Recovery Improves with Sample Size', fontsize=14)
plt.axhline(y=0, color='green', linestyle='--', alpha=0.5, label='Perfect recovery')
plt.legend()
plt.tight_layout()
plt.show()

---

## Part 4: LiNGAM for Non-Gaussian Data

### Key Insight: Non-Gaussianity Enables Unique Identification

For Gaussian data, $X \to Y$ and $Y \to X$ are **indistinguishable** (symmetric distributions).

For non-Gaussian data, ICA can **uniquely identify** the mixing matrix → causal order.

### Structural Equation Model

$$X = B^T X + \varepsilon$$

where $\varepsilon$ has **non-Gaussian** independent components.

LiNGAM recovers $B$ via Independent Component Analysis (ICA).

In [ ]:
# Generate non-Gaussian data (Laplace noise)
data_laplace, B_true = generate_dag_data(
    diamond_dag, n_samples=1000, noise_type="laplace", seed=42
)

# Check non-Gaussianity
ng_result = check_non_gaussianity(data_laplace)

print("=== Non-Gaussianity Check ===")
print(f"Overall non-Gaussian: {ng_result.is_non_gaussian}")
print(f"\nPer-variable kurtosis (excess):")
for i, k in enumerate(ng_result.kurtosis):
    status = "✓ Non-Gaussian" if abs(k) > 0.5 else "~ Gaussian"
    print(f"  X{i}: {k:>6.2f} {status}")

print("\n→ Laplace distribution has excess kurtosis of 3.0")

### DirectLiNGAM Algorithm

DirectLiNGAM finds causal order by iteratively identifying the **exogenous** variable:

1. Find variable with **minimum** residual dependence on others
2. Mark as next in causal order
3. Regress out its effect
4. Repeat

In [ ]:
# Run DirectLiNGAM
lingam_result = direct_lingam(data_laplace)

print("=== DirectLiNGAM Result ===")
print(f"Causal order: {lingam_result.causal_order}")
print(f"\nInterpretation:")
for i, var in enumerate(lingam_result.causal_order):
    print(f"  Position {i}: X{var}")

# Compare to true topological order
true_order = diamond_dag.topological_order()
print(f"\nTrue topological order: {true_order}")

In [ ]:
# Examine recovered DAG
print("=== Recovered DAG ===")
print(f"Edges: {lingam_result.dag.n_edges}")
print("\nEdges:")
for i, j in lingam_result.dag.edges():
    print(f"  X{i} → X{j}")

# Compare coefficient matrix
print(f"\nRecovered B matrix:")
print(np.round(lingam_result.adjacency_matrix, 2))
print(f"\nTrue B matrix:")
print(np.round(B_true, 2))

### Bootstrap Confidence

In [ ]:
# Bootstrap for edge stability
bootstrap_result = bootstrap_lingam(data_laplace, n_bootstrap=100, seed=42)

print("=== Bootstrap Edge Stability ===")
print("Edge frequencies (% of bootstrap samples):")

edge_probs = bootstrap_result.edge_probabilities
for i in range(4):
    for j in range(4):
        if edge_probs[i, j] > 0.1:
            print(f"  X{i} → X{j}: {edge_probs[i, j]*100:.0f}%")

print("\n→ High frequency = stable edge")

---

## Part 5: Choosing Between Methods

### Decision Tree

```
Is data distribution known?
├── Yes, Gaussian → PC Algorithm
├── Yes, Non-Gaussian → LiNGAM (preferred)
└── Unknown
    ├── Check kurtosis (check_non_gaussianity)
    ├── Non-Gaussian → LiNGAM
    └── Gaussian-ish → PC Algorithm
```

### Comparison Table

| Aspect | PC Algorithm | LiNGAM |
|--------|--------------|--------|
| **Assumption** | Faithfulness | Non-Gaussianity |
| **Output** | CPDAG (equivalence class) | Unique DAG |
| **Identifiability** | Partial | Full |
| **Sample size** | 200+ | 500+ (ICA needs more) |
| **Scalability** | O(p² × n) | O(p² × n) |
| **Latent confounders** | Not handled | Not handled |

In [ ]:
# Compare both methods on same data

print("=== Method Comparison ===")
print("\n--- Gaussian Data ---")
data_gauss, _ = generate_dag_data(diamond_dag, n_samples=1000, noise_type="gaussian", seed=42)

pc_gauss = pc_algorithm(data_gauss, alpha=0.01)
lingam_gauss = direct_lingam(data_gauss)

shd_pc_gauss = compute_shd(pc_gauss.cpdag, true_cpdag)
print(f"PC SHD: {shd_pc_gauss}")
print(f"LiNGAM edges: {lingam_gauss.dag.n_edges} (may be unreliable on Gaussian)")

print("\n--- Non-Gaussian Data ---")
pc_laplace = pc_algorithm(data_laplace, alpha=0.01)

shd_pc_laplace = compute_shd(pc_laplace.cpdag, true_cpdag)
print(f"PC SHD: {shd_pc_laplace}")
print(f"LiNGAM edges: {lingam_result.dag.n_edges}")
print(f"LiNGAM recovers UNIQUE DAG (not equivalence class)")

---

## Part 6: Practical Example - 10-Node Network

Let's apply both methods to a larger, more realistic network.

In [ ]:
# Generate larger DAG
large_dag = generate_random_dag(n_vars=10, edge_prob=0.25, seed=123)

print(f"=== 10-Node DAG ===")
print(f"Nodes: {large_dag.n_nodes}")
print(f"Edges: {large_dag.n_edges}")
print(f"\nEdges:")
for i, j in sorted(large_dag.edges()):
    print(f"  X{i} → X{j}")

In [ ]:
# Generate data (non-Gaussian for LiNGAM)
data_large, B_large = generate_dag_data(
    large_dag, n_samples=2000, noise_type="laplace", seed=123
)

print(f"Data shape: {data_large.shape}")

In [ ]:
# PC Algorithm
import time

start = time.time()
pc_large = pc_algorithm(data_large, alpha=0.01)
pc_time = time.time() - start

large_cpdag = dag_to_cpdag(large_dag)
shd_pc = compute_shd(pc_large.cpdag, large_cpdag)
prec, rec, f1_pc = skeleton_f1(pc_large.skeleton, large_dag)

print("=== PC Algorithm Results ===")
print(f"Time: {pc_time:.2f}s")
print(f"CI tests: {pc_large.n_ci_tests}")
print(f"SHD: {shd_pc}")
print(f"Skeleton F1: {f1_pc:.3f}")

In [ ]:
# DirectLiNGAM
start = time.time()
lingam_large = direct_lingam(data_large)
lingam_time = time.time() - start

# Compare causal order
true_order = large_dag.topological_order()
estimated_order = lingam_large.causal_order

print("=== DirectLiNGAM Results ===")
print(f"Time: {lingam_time:.2f}s")
print(f"Edges recovered: {lingam_large.dag.n_edges}")
print(f"\nCausal order:")
print(f"  Estimated: {estimated_order}")
print(f"  True:      {true_order}")

# Order accuracy
order_matches = sum(1 for i, v in enumerate(estimated_order) if v == true_order[i])
print(f"\nPosition matches: {order_matches}/{len(true_order)}")

In [ ]:
# Summary comparison
print("=== Method Comparison Summary ===")
print(f"{'Metric':>20} {'PC':>12} {'LiNGAM':>12}")
print("-" * 46)
print(f"{'Time (s)':>20} {pc_time:>12.2f} {lingam_time:>12.2f}")
print(f"{'Skeleton F1':>20} {f1_pc:>12.3f} {'N/A':>12}")
print(f"{'SHD':>20} {shd_pc:>12} {'N/A':>12}")
print(f"{'Unique DAG':>20} {'No':>12} {'Yes':>12}")
print(f"{'Edges recovered':>20} {pc_large.cpdag.n_directed_edges + pc_large.cpdag.n_undirected_edges:>12} {lingam_large.dag.n_edges:>12}")
print(f"{'True edges':>20} {large_dag.n_edges:>12} {large_dag.n_edges:>12}")

---

## Summary

### Key Takeaways

1. **PC Algorithm** learns Markov equivalence class (CPDAG) via conditional independence tests
2. **LiNGAM** exploits non-Gaussianity for unique DAG identification
3. **Alpha** controls PC's edge inclusion trade-off
4. **Sample size** dramatically affects recovery quality
5. **Check non-Gaussianity** before choosing LiNGAM

### Limitations

- Neither method handles **latent confounders** (use FCI for that)
- Both assume **causal sufficiency** (no hidden common causes)
- **Faithfulness violations** can cause PC errors
- **Near-Gaussianity** can cause LiNGAM errors

### References

1. **Spirtes, Glymour, Scheines (2000)** - *Causation, Prediction, and Search*
2. **Shimizu et al. (2006)** - "A linear non-Gaussian acyclic model for causal discovery"
3. **Shimizu et al. (2011)** - "DirectLiNGAM: A direct method for learning a linear non-Gaussian structural equation model"
4. **Meek (1995)** - "Causal inference and causal explanation with background knowledge"

---

*Created: Session 134 - Causal Inference Mastery*